# 04 · Evaluación y comparación
Se priorizan Recall, Precision, F1 y ROC-AUC. Accuracy no se usa como criterio único.

> **Advertencia:** mientras no se disponga de datos institucionales, estos notebooks usan el dataset **SINTÉTICO** `data/sample/synthetic_training_dataset.csv`. Los resultados sirven sólo para verificar técnicamente el pipeline y **no deben reportarse como hallazgos académicos**.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent / "backend"))  # sipd_ml vive en backend/
import pandas as pd
from sipd_ml import config
pd.set_option("display.max_columns", 30)

In [2]:
from sipd_ml.evaluation import compare_models, evaluate_estimator
from sipd_ml.preprocessing import prepare_training_frame
from sipd_ml.training import split_and_preprocess, train_candidates
df = pd.read_csv(config.SYNTHETIC_DATASET_PATH)
split = split_and_preprocess(*prepare_training_frame(df))
candidates = train_candidates(split)
results = {c.algorithm: evaluate_estimator(c.estimator, split.x_test, split.y_test) for c in candidates}
ranking = pd.DataFrame(compare_models(results, 'f1')).set_index('algorithm')
ranking[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]

,accuracy,precision,recall,f1,roc_auc
algorithm,,,,,
logistic_regression,0.779,0.4824,0.5802,0.5268,0.8084
random_forest,0.769,0.4639,0.5755,0.5137,0.8014
xgboost,0.770,0.4646,0.5566,0.5064,0.7883


In [3]:
pd.DataFrame({k: v['confusion_matrix'] for k, v in results.items()}).T

,tn,fp,fn,tp
logistic_regression,656,132,89,123
random_forest,647,141,90,122
xgboost,652,136,94,118


El umbral de clasificación usado para estas métricas es `config.CLASSIFICATION_THRESHOLD` (0,30 por defecto, coherente con el inicio del nivel de riesgo Medio).